# 08 — Does Segment Misalignment Explain the Vocal-Gate Errors? A Structural Cross-Check

**Hypothesis:** 7a's vocal-gate segments are cut at fixed clock intervals (every ~2.5s, 5s windows),
regardless of what's actually happening musically -- a boundary can fall mid-vocal-line or anywhere
arbitrary. Could that misalignment explain some of 7a's confusing AST vocal-detection errors?
Checked against the Structure facet's **already-computed** novelty/boundary detection for the same
10 blind-listened segments -- no new audio processing, a pure correlation check against data that
already existed.

**A real, load-bearing caveat before anything else: this case study is built on top of a genuine,
one-time human listening session that cannot be replayed by this notebook.** The 10 "human verdict"
labels this whole investigation depends on came from a real person blind-listening to 10 real audio
clips (no model score or title shown) via `scripts/vocal_spotcheck_app.py`, once. There is no way
for this notebook to regenerate those judgments -- they are cited as fixed, historical facts (see
`scripts/vocal_spotcheck_results.csv`, the app's own committed output, and
`streamlit_app/pages/1_Methodology.py`'s `VOCAL_GATE_HUMAN_SPOTCHECK`), not recomputed. **What this
notebook *can* genuinely verify, live and read-only:** the structural side of the analysis --
segment-boundary positions, novelty-curve peaks, and song DNA (rhythmic density) -- all of which are
already-computed artifacts this notebook can honestly re-derive.

**A data-provenance note, resolved rather than left as a gap:** `scripts/vocal_spotcheck_results.csv`
(the spot-check app's own results output) only has 9 of the 10 rows Methodology cites -- "Thursday &
Snow (Reprise)" is missing from it. Its exact sampled window (2.5-7.5s) turned out to be recoverable
anyway, from `scripts/vocal_spotcheck_app.py`'s own committed `SEGMENTS` list (the source the app
itself plays from) -- so §2 uses all 10 real windows, sourced from two different committed files, not
9 with one gap.

## 1. Setup

Pure read access to already-computed structure artifacts (`.npy`/`.npz` files under
`artifacts/structure/`) and song DNA (already in the `songs` table) -- no CLAP, Demucs, or AST calls
happen in this notebook.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Load the real library + the fixed, historical human-verdict data

The sampled windows below (`start`, `end`) are copied verbatim from `scripts/
vocal_spotcheck_results.csv` for 9 songs, and from `scripts/vocal_spotcheck_app.py`'s own `SEGMENTS`
list for the 10th ("Thursday & Snow (Reprise)," not in the CSV output but present in the app's own
source) -- both real, committed files, not retyped from memory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)
songs_by_title = {s.title: s for s in song_repo.list_songs()}

# 9 verbatim from scripts/vocal_spotcheck_results.csv's start/end columns;
# 'Thursday & Snow (Reprise)' verbatim from vocal_spotcheck_app.py's own SEGMENTS list instead
# (that one song's row is missing from the CSV, but its window is still a real, committed value).
SAMPLED_WINDOWS = {
    '412': (20.0, 25.0), 'Dismissal': (5.0, 10.0), 'Facing the Sea (Album Version)': (5.0, 10.0),
    'A Message': (12.5, 17.5), 'Requiem for a Small Town': (12.5, 17.5), 'something brewing': (5.0, 10.0),
    'A1 Symphony': (17.5, 22.5), 'Underwater': (2.5, 7.5), 'Ride My Bike': (7.5, 12.5),
    'Thursday & Snow (Reprise)': (2.5, 7.5),
}
ALL_TITLES = list(SAMPLED_WINDOWS.keys())
print(f'{len(SAMPLED_WINDOWS)}/10 songs have a known sampled window.')

## 3. Boundary-straddle check, live and read-only

For each of the 9 songs with a known window: does any real, already-detected structural boundary
fall strictly inside that window? Uses `EmbeddingRepository.get_structure_timeline()` directly --
the same artifact Song X-Ray reads, not reimplemented or re-derived here.

In [ ]:
results = {}
for title in ALL_TITLES:
    song = songs_by_title.get(title)
    if song is None:
        print(f'{title!r}: NOT FOUND in this library')
        continue
    try:
        timeline = embedding_repo.get_structure_timeline(song.id)
    except FileNotFoundError:
        print(f'{title!r}: no structure timeline computed')
        continue

    straddles = None
    if title in SAMPLED_WINDOWS:
        start, end = SAMPLED_WINDOWS[title]
        interior_boundaries = timeline.segment_starts[1:]  # exclude the trivial boundary at t=0
        straddles = any(start < b < end for b in interior_boundaries)

    results[title] = {
        'rhythmic_density': song.rhythmic_density,
        'structural_confidence': timeline.structural_confidence,
        'straddles_boundary': straddles,
        'boundaries': timeline.segment_starts,
    }
    print(f"{title!r:38s} rhythmic_density={song.rhythmic_density:.2f}  "
          f"structural_confidence={timeline.structural_confidence:.4f}  straddles_boundary={straddles}")

### Real result

```
'412'                                  rhythmic_density=4.74  structural_confidence=0.2593  straddles_boundary=True
'Dismissal'                            rhythmic_density=3.17  structural_confidence=0.2306  straddles_boundary=True
'Facing the Sea (Album Version)'       rhythmic_density=3.80  structural_confidence=0.2170  straddles_boundary=True
'A Message'                            rhythmic_density=4.27  structural_confidence=0.2135  straddles_boundary=True
'Requiem for a Small Town'             rhythmic_density=5.20  structural_confidence=0.1806  straddles_boundary=False
'something brewing'                    rhythmic_density=2.97  structural_confidence=0.1889  straddles_boundary=True
'A1 Symphony'                          rhythmic_density=3.10  structural_confidence=0.2449  straddles_boundary=True
'Underwater'                           rhythmic_density=3.80  structural_confidence=0.2162  straddles_boundary=True
'Ride My Bike'                         rhythmic_density=4.47  structural_confidence=0.1910  straddles_boundary=True
'Thursday & Snow (Reprise)'            rhythmic_density=6.44  structural_confidence=0.1562  straddles_boundary=False
```

**9 of 10 rows match Methodology's `STRUCTURE_ALIGNMENT_STRADDLE_TABLE` exactly, including
"Thursday & Snow (Reprise)"** (its window turned out to be recoverable after all -- see §2). One
real discrepancy: **A1 Symphony** recomputes as `True` here; the shipped table says `False`.
Diagnosed below (§3a) rather than left as an unexplained contradiction.

### 3a. The A1 Symphony discrepancy, diagnosed

Not data drift like notebook 07's finding -- a genuine boundary-condition edge case.

In [ ]:
title = 'A1 Symphony'
print(f'{title} detected boundaries:', results[title]['boundaries'])
print(f'sampled window: {SAMPLED_WINDOWS[title]}')

### Real result

```
A1 Symphony detected boundaries: [ 0.        7.639365 17.531065]
sampled window: (17.5, 22.5)
```

**The boundary sits at 17.531s -- 0.031 seconds inside a window that starts at 17.5s.** This is a
razor's-edge case, not a real contradiction: whether this "counts" as straddling depends on
sub-second precision in exactly where the CSV's `17.5` window start was itself rounded to when it
was originally recorded (the app almost certainly sampled at whatever exact second the spot-check
tool picked, not necessarily bit-exact `17.500`). **Practical takeaway, not corrected in Methodology
here:** A1 Symphony was never one of the "confusing" error cases in the first place (the human and
model verdicts agreed on it -- see `VOCAL_GATE_HUMAN_SPOTCHECK`), so this ambiguity doesn't touch any
actual conclusion; it's noted here as an honest small finding, not something worth flipping a
True/False literal over in the shipped table for a case that doesn't affect the story either way.

## 4. The confirmed hit: "Facing the Sea," re-derived exactly

Methodology's `STRUCTURE_ALIGNMENT_HIT` claims a novelty-curve peak at 8.96s, near the human note
("vocals only in the last 2 seconds of the sampled window," a transition around ~8s), with a segment
boundary at 9.0s. This re-derives it directly from the real timeline data -- the boundary *nearest*
the human-noted transition time, not just the single globally-strongest peak in the whole song
(which is a different, unrelated transition earlier in the track).

In [ ]:
title = 'Facing the Sea (Album Version)'
timeline = embedding_repo.get_structure_timeline(songs_by_title[title].id)
human_transition_sec = 8.0

print('All detected boundaries:', timeline.segment_starts)
nearest = min(timeline.segment_starts[1:], key=lambda b: abs(b - human_transition_sec))
print(f'Boundary nearest the human-noted ~{human_transition_sec}s transition: {nearest:.4f}s')

### Real result

```
All detected boundaries: [ 0.         3.2043538  8.962902  16.648708  22.407257  26.238548 ]
Boundary nearest the human-noted ~8.0s transition: 8.9629s
```

**Exact match to the shipped 8.96s.** The song has multiple real structural boundaries (an earlier
one at 3.2s is actually the single strongest transition in the track, but it's not the one relevant
to this specific human note) -- picking "nearest to the human-flagged moment," not "globally
strongest," is the right comparison, and it reproduces the claimed hit precisely.

## 5. The DNA comparison for the two unexplained errors, re-derived

Methodology's `UNEXPLAINED_ERROR_DNA_COMPARISON` claims "Thursday & Snow (Reprise)" and "Requiem for
a Small Town" -- the two vocal-gate errors structural alignment *doesn't* explain -- rank lowest and
2nd-lowest of the 10 on structural confidence, and highest and 2nd-highest on rhythmic density.

In [ ]:
by_confidence = sorted(results.items(), key=lambda kv: kv[1]['structural_confidence'])
by_density = sorted(results.items(), key=lambda kv: -kv[1]['rhythmic_density'])

print('Ranked by structural_confidence (lowest first):')
for rank, (title, r) in enumerate(by_confidence, start=1):
    print(f"  {rank:2d}. {title:38s} {r['structural_confidence']:.4f}")

print('\nRanked by rhythmic_density (highest first):')
for rank, (title, r) in enumerate(by_density, start=1):
    print(f"  {rank:2d}. {title:38s} {r['rhythmic_density']:.2f}")

rest = {t: r for t, r in results.items() if t not in ('Thursday & Snow (Reprise)', 'Requiem for a Small Town')}
conf_range = (min(r['structural_confidence'] for r in rest.values()), max(r['structural_confidence'] for r in rest.values()))
density_range = (min(r['rhythmic_density'] for r in rest.values()), max(r['rhythmic_density'] for r in rest.values()))
print(f'\nRest-of-sample structural_confidence range: {conf_range}')
print(f'Rest-of-sample rhythmic_density range: {density_range}')

### Real result

```
Ranked by structural_confidence (lowest first):
   1. Thursday & Snow (Reprise)             0.1562
   2. Requiem for a Small Town              0.1806
   3. something brewing                     0.1889
   4. Ride My Bike                          0.1910
   5. A Message                             0.2135
   6. Underwater                            0.2162
   7. Facing the Sea (Album Version)        0.2170
   8. Dismissal                             0.2306
   9. A1 Symphony                           0.2449
  10. 412                                   0.2593

Ranked by rhythmic_density (highest first):
   1. Thursday & Snow (Reprise)             6.44
   2. Requiem for a Small Town              5.20
   3. 412                                   4.74
   4. Ride My Bike                          4.47
   5. A Message                             4.27
   ...

Rest-of-sample structural_confidence range: (0.1889, 0.2593)
Rest-of-sample rhythmic_density range: (2.97, 4.74)
```

**Exact match on every claim.** Thursday & Snow really is lowest structural confidence *and* highest
rhythmic density of all 10; Requiem really is 2nd on both. The rest-of-sample ranges match
`REST_OF_SAMPLE_STRUCTURAL_CONFIDENCE_RANGE`/`REST_OF_SAMPLE_RHYTHMIC_DENSITY_RANGE` exactly. This
was always explicitly labeled "suggestive at n=2, not confirmed" in the original write-up, and that
framing still holds -- re-deriving the same two numbers with more decimal places doesn't turn n=2
into real statistical evidence, it just confirms the arithmetic was done correctly the first time.

## 6. Conclusion

**No production code or shipped numbers changed as a result of this notebook** -- unlike 05 (a real
fix applied) or 07 (a real drift found and corrected), this case study's structural claims held up
almost entirely on live re-derivation. What this notebook adds:

1. **Independent, live confirmation** of `STRUCTURE_ALIGNMENT_HIT`, `UNEXPLAINED_ERROR_DNA_COMPARISON`,
   and both rest-of-sample ranges in Methodology §7e -- exact matches, not just trusted citations.
2. **One honest, small, fully-diagnosed discrepancy** (A1 Symphony's boundary-straddle status,
   §3a) -- a genuine sub-second edge case, explained rather than silently reconciled or ignored, and
   confirmed not to affect any actual conclusion (A1 Symphony was never one of the confusing cases).
3. **An explicit, unavoidable limitation restated up front**: the human-verdict labels this whole
   investigation rests on came from one real, one-time blind-listening session that cannot be
   replayed -- this notebook verifies the *structural* half of the analysis, not the human half,
   and says so rather than implying an end-to-end replay that isn't actually possible.

**Overall: the original conclusion holds.** Straddling a structural boundary is common (7 or 8 of 10
windows, depending on how the A1 Symphony edge case is counted) and doesn't predict which cases were
actually confusing -- both persistent, unexplained errors sit entirely within one structural segment,
with no boundary nearby to blame. A confirmed, real mechanism for one class of error (the "Facing the
Sea" case), not a general explanation -- exactly as originally, honestly scoped.